In [1]:
# ============================================================
# Импорт библиотек
# ============================================================
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

In [2]:
# ============================================================
# 1. Список файлов и настройки
# ============================================================
FILES = [
    "/content/75_chelyabinsk.csv",
    "/content/37_kurgan.csv",
    "/content/33_kyrov.csv"
]

TARGET_COLS = ["actual_consumption", "actual_generation"]

# Горизонт прогноза (часы). 1 год = 8760 ч (не високосный)
FORECAST_HORIZON = 24 * 365

# Параметры SARIMAX (сезонность — суточная, 24 ч)
ORDER = (1, 1, 1)
SEASONAL_ORDER = (1, 1, 1, 24)

# Пиковые часы — определяем как часы, где потребление/генерация
# выше 90-го перцентиля (можно настроить)
PEAK_PERCENTILE = 90

In [3]:
# ============================================================
# 2. Вспомогательные функции
# ============================================================

def load_and_prepare(path: str) -> pd.DataFrame:
    """Загрузка CSV, парсинг даты, установка индекса, ресемплинг на час."""
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").set_index("date")

    # Приводим к часовому индексу (на случай пропусков)
    df = df.asfreq("H")
    # Интерполяция пропусков (линейная) — безопасно для рядов
    df[TARGET_COLS] = df[TARGET_COLS].interpolate(method="linear")
    df = df.dropna(subset=TARGET_COLS)
    return df


def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    """Добавляем признаки выходных и праздников РФ (упрощённый набор)."""
    idx = df.index
    df = df.copy()
    df["is_weekend"] = (idx.dayofweek >= 5).astype(int)

    # Основные праздники РФ (можно расширить)
    holidays = {
        (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8),  # Новый год
        (2, 23),   # День защитника Отечества
        (3, 8),    # 8 марта
        (5, 1),    # Праздник весны и труда
        (5, 9),    # День Победы
        (6, 12),   # День России
        (11, 4),   # День народного единства
    }
    df["is_holiday"] = [
        1 if (d.month, d.day) in holidays else 0 for d in idx
    ]
    # Час суток (для сезонности можно добавить, но SARIMAX с m=24 учтёт сам)
    df["hour"] = idx.hour
    return df


def fit_sarimax(train: pd.DataFrame, target: str, exog_cols: list):
    """Обучение SARIMAX на одном целевом ряду."""
    y = train[target]
    X = train[exog_cols] if exog_cols else None
    model = SARIMAX(
        y,
        exog=X,
        order=ORDER,
        seasonal_order=SEASONAL_ORDER,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    res = model.fit(disp=False)
    return res


def predict_peak_hours(series: pd.Series, percentile: int = PEAK_PERCENTILE) -> pd.DatetimeIndex:
    """Возвращает индексы пиковых часов (выше percentile)."""
    thr = np.percentile(series.values, percentile)
    return series[series >= thr].index


def peak_accuracy(actual: pd.Series, pred: pd.Series, percentile: int = PEAK_PERCENTILE) -> float:
    """
    Точность прогнозирования пиковых часов, %.
    Считаем долю фактических пиковых часов, которые модель
    также отнесла к пиковым (по прогнозу), т.е. recall по пикам.
    """
    act_peaks = set(predict_peak_hours(actual, percentile))
    pred_peaks = set(predict_peak_hours(pred, percentile))
    if len(act_peaks) == 0:
        return np.nan
    hit = len(act_peaks & pred_peaks)
    return hit / len(act_peaks) * 100.0


def rolling_forecast(
    df: pd.DataFrame,
    target: str,
    exog_cols: list,
    horizon: int = FORECAST_HORIZON,
    step: int = 24,
    initial_train_end: str = "2025-01-01",
):
    """
    Режим скользящего окна (аналог split.py):
    - обучаемся на данных до initial_train_end;
    - прогнозируем step часов вперёд;
    - сдвигаем окно на step часов и повторяем, пока не покроем horizon.
    Возвращает Series прогнозов с DatetimeIndex.
    """
    forecasts = []
    train_end = pd.Timestamp(initial_train_end)
    # Если в данных нет такого периода — берём последнюю доступную дату минус horizon
    if df.index.max() < train_end:
        train_end = df.index.max() - pd.Timedelta(hours=horizon)

    total_steps = int(np.ceil(horizon / step))
    for i in range(total_steps):
        train = df.loc[:train_end]
        if len(train) < 24 * 7:  # слишком мало данных
            break
        try:
            res = fit_sarimax(train, target, exog_cols)
        except Exception as e:
            print(f"  [!] Ошибка обучения на шаге {i}: {e}")
            train_end += pd.Timedelta(hours=step)
            continue

        # Экзогенные признаки для будущего периода
        future_idx = pd.date_range(
            start=train_end + pd.Timedelta(hours=1),
            periods=step,
            freq="H",
        )
        future_exog = pd.DataFrame(index=future_idx)
        future_exog["is_weekend"] = (future_idx.dayofweek >= 5).astype(int)
        holidays = {
            (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8),
            (2, 23), (3, 8), (5, 1), (5, 9), (6, 12), (11, 4),
        }
        future_exog["is_holiday"] = [
            1 if (d.month, d.day) in holidays else 0 for d in future_idx
        ]
        future_exog["hour"] = future_idx.hour

        X_future = future_exog[exog_cols] if exog_cols else None
        try:
            fc = res.forecast(steps=step, exog=X_future)
            fc.index = future_idx
        except Exception as e:
            print(f"  [!] Ошибка прогноза на шаге {i}: {e}")
            train_end += pd.Timedelta(hours=step)
            continue

        forecasts.append(fc)
        train_end += pd.Timedelta(hours=step)

    if not forecasts:
        return pd.Series(dtype=float)
    return pd.concat(forecasts)

In [ ]:
# ============================================================
# 3. Основной цикл по файлам
# ============================================================

all_results = {}

for fname in FILES:
    print("=" * 70)
    print(f"Файл: {fname}")
    print("=" * 70)

    if not os.path.exists(fname):
        print(f"  [!] Файл {fname} не найден — пропуск.")
        continue

    df = load_and_prepare(fname)
    df = add_calendar_features(df)
    exog_cols = ["is_weekend", "is_holiday", "hour"]

    file_results = {}
    for target in TARGET_COLS:
        print(f"\n--- Целевой ряд: {target} ---")
        fc = rolling_forecast(df, target, exog_cols)

        # Фактические значения на периоде прогноза
        actual = df[target].reindex(fc.index).dropna()
        pred = fc.reindex(actual.index).dropna()
        common = actual.index.intersection(pred.index)
        actual, pred = actual.loc[common], pred.loc[common]

        if len(actual) == 0:
            print("  [!] Нет пересечения прогноза и факта — пропуск метрик.")
            continue

        mae = mean_absolute_error(actual, pred)
        rmse = np.sqrt(mean_squared_error(actual, pred))
        r2 = r2_score(actual, pred)
        peak_acc = peak_accuracy(actual, pred)

        file_results[target] = {
            "forecast": pred,
            "actual": actual,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "PeakAccuracy_%": peak_acc,
        }

        print(f"  MAE  = {mae:.3f}")
        print(f"  RMSE = {rmse:.3f}")
        print(f"  R^2  = {r2:.4f}")
        print(f"  Точность пиковых часов = {peak_acc:.2f}%")

    all_results[fname] = file_results

Файл: /content/75_chelyabinsk.csv

--- Целевой ряд: actual_consumption ---


In [ ]:
# ============================================================
# 4. Сводная таблица метрик
# ============================================================
print("\n" + "=" * 70)
print("СВОДНАЯ ТАБЛИЦА МЕТРИК")
print("=" * 70)
rows = []
for fname, res in all_results.items():
    for target, m in res.items():
        rows.append({
            "file": fname,
            "target": target,
            "MAE": m["MAE"],
            "RMSE": m["RMSE"],
            "R2": m["R2"],
            "PeakAccuracy_%": m["PeakAccuracy_%"],
        })
summary = pd.DataFrame(rows)
if not summary.empty:
    print(summary.to_string(index=False))
    summary.to_csv("sarimax_metrics_summary.csv", index=False)

In [ ]:
# ============================================================
# 5. Визуализация
# ============================================================
for fname, res in all_results.items():
    for target, m in res.items():
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.plot(m["actual"].index, m["actual"].values, label="Факт", lw=1)
        ax.plot(m["forecast"].index, m["forecast"].values, label="Прогноз SARIMAX", lw=1, alpha=0.8)

        # Отмечаем пиковые часы факта
        act_peaks = predict_peak_hours(m["actual"])
        ax.scatter(act_peaks, m["actual"].loc[act_peaks],
                   color="red", s=8, label="Пиковые часы (факт)", zorder=5)

        ax.set_title(f"{fname} — {target}\n"
                     f"MAE={m['MAE']:.2f}, RMSE={m['RMSE']:.2f}, "
                     f"R²={m['R2']:.3f}, Пик.точность={m['PeakAccuracy_%']:.1f}%")
        ax.set_xlabel("Дата")
        ax.set_ylabel("Значение")
        ax.legend(loc="upper right")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        fig.autofmt_xdate()
        plt.tight_layout()
        safe_name = f"{os.path.splitext(fname)[0]}_{target}_forecast.png"
        plt.savefig(safe_name, dpi=120)
        plt.show()
        print(f"  График сохранён: {safe_name}")

print("\nГотово.")